# E1.1 · Why point-in-time control testing fails for AI

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.0 · Start here — what AI governance means](https://spbreed.github.io/cyber-commons/lessons/E1.0.html)**.

| | |
|---|---|
| Tools used | promptfoo |

## What this lesson is

**What it covers.** Change a prompt and show the control evidence going stale in real time.

**Why a security engineer needs it.** An annual review certifies nothing about a system that changed on Tuesday. The control it builds is: continuous assurance; control effectiveness redefined for probabilistic systems.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

You tested the control in March and signed the assertion. The prompt changed in April, the model in May, and the tool scope in June. The assertion is still on file and has not described anything real since the day it was written.

> **At CyberTravels.** The control test that passed in March described a CyberTravels with no payments scope, no repository access and no vector store. Nothing about it was wrong; everything about it is stale.

## 2 · The framework

```
   march      test the control, sign the assertion
   april      the prompt changes
   may        the model version changes
   june       the tool scope changes
   december   the assertion is still on file

   point-in-time assurance for a system that changes between tests
   describes a system that no longer exists
```

Classical control testing has a simple shape: a control is designed, an auditor
tests it once or twice a year, and a passing test is recorded for the period.

That works when the thing being tested changes only through a process that
generates evidence. For an agent, the four things that change its behaviour are:

- the **model version** — changed by your provider, possibly without notice,
- the **prompt** — edited in a console,
- the **tool manifest** — a config change,
- the **approval settings** — a toggle in an admin UI.

None of them is a code change. None generates a change record. All of them
invalidate the conditions the control was tested under.

The honest consequence is that a control tested six months ago is not passing —
it is **unevidenced**, which is a third state most GRC tooling cannot represent.
Introducing that third state is the whole of this lesson.

## 3 · The control — a freshness window per control, derived from drift

The window is not an audit-calendar choice. It comes from **how fast the thing the control tests actually changes.**

## 4 · What replaces the annual test, as a skill

If the window is short, something has to re-run inside it, and that something is an attestation: collect each control's verdict, resolve every evidence pointer, compute drift against the image digest and the tool manifest, and sign the result. Two rules in the procedure are the whole difference between an attestation and a slide — a missing verdict is not a pass, and a capped verdict does not get raised because the other evidence looked good. This is the file in this repository:

In [ ]:
# skills/attestation/attestation-signer-lifecycle/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: attestation-signer-lifecycle
description: >-
  Bundle control evidence, compute verdicts, and produce a signed verifiable
  attestation for a deployment, re-attesting on drift. Use to emit or verify
  a control attestation, to sign evidence into an in-toto envelope, or to
  decide whether drift requires re-attestation.
allowed-tools: Bash, Read, Write
---

# Attestation Signer Lifecycle

**Controls:** All — the artifact

## Use the existing framework

Do not invent a format. Wrap the evidence in an **in-toto style statement**
inside a signed envelope, with the subject naming the deployment and its
digests, and a typed predicate carrying the control verdicts. Express the
predicate body in an assessment-results vocabulary so the verdicts map to
control catalogues and an auditor can consume them.

The roles are worth naming explicitly: this skill set is the **attester**
producing evidence, a separate service is the **verifier** appraising it
against reference values, and the deployment gate is the **relying party**
applying policy. Keep them separate — an attester that also decides whether it
passed is not an attestation.

## Procedure

1. **Collect every sub-skill verdict.** A missing verdict is not a pass. If a
   skill did not run, the control is `UNKNOWN` and the attestation says so.
2. **Apply the confidence ceilings.** Sandbox-egress and injection-screening are
   capped at `PARTIAL` by their own skills; the signer must refuse to raise them.
3. **Resolve every evidence pointer.** A URI that does not resolve is a broken
   attestation, not a cosmetic issue.
4. **Attach framework mappings**, so one artefact answers several catalogues.
5. **Compute drift.** Compare image digest, configuration hashes and tool
   description hashes against the previous attestation. Any change triggers
   re-attestation — the tool-description hash specifically defends against a
   server mutating a tool after approval.
6. **Sign**, and store alongside the image or in a transparency log.

## Output contract

```json
{
  "subject": [{"name": "deployment_id", "digest": {"image": "sha256:…", "repo": "str"}}],
  "predicate_type": "str",
  "predicate": {
    "deployment_id": "str", "evaluated_at": "str", "evaluator_version": "str",
    "controls": [
      {"id": "C1_default_deny_least_privilege",
       "verdict": "PASS|FAIL|PARTIAL|UNKNOWN",
       "confidence": "HIGH|PARTIAL",
       "evidence": [{"skill": "str", "uri": "str", "hash": "str"}],
       "findings": ["str"]}
    ],
    "framework_mappings": {"owasp_llm": ["str"], "owasp_agentic": ["str"],
                           "atlas": ["str"], "nist_ai_rmf": ["str"], "iso_42001": ["str"]},
    "drift": {"since": "str", "changed": ["str"]}
  },
  "signatures": [{"keyid": "str", "sig": "str"}]
}
```

## Failure modes

- **Reading a missing attestation as a pass.** The relying party must **fail
  closed**: a signed file can be deleted, and absence is not evidence.
- **Raising a capped verdict** because the other evidence looked good.
- **Signing without resolving evidence pointers**, which produces a
  tamper-evident document full of dead links.
- **Attesting once.** Without drift-triggered re-attestation the artefact
  describes a deployment that may no longer exist.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, os, shutil, sys

# Make the shared runtime importable, then import it. On Kaggle an attached
# kernel is mounted as __script__.py — not on sys.path and not named after the
# kernel — so copy it to the name it is imported by. Locally it is already a
# file of that name in the repository.
_k = glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py", recursive=True)
if _k:
    shutil.copy(_k[0], "cyber_commons_skill_runtime.py")
sys.path[:0] = [".", "skills/_runtime", "../skills/_runtime", "../../skills/_runtime"]

from cyber_commons_skill_runtime import run_skill

# Split skills/attestation/attestation-signer-lifecycle/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

In [ ]:
# skills/attestation/attestation-signer-lifecycle/scripts/attestation_signer_lifecycle.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Read the same evidence as a point-in-time test and as a continuous one, and derive a freshness window per control from observed drift.

This is the executable half of the `attestation-signer-lifecycle` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import time
from dataclasses import dataclass, field

now = time.time(); DAY = 86400

@dataclass
class ControlTest:
    cid: str
    passed: bool
    evidence: str
    tested_at: float
    valid_for_days: float = 30

    def age_days(self, at): return (at - self.tested_at) / DAY
    def point_in_time(self, at): return "PASS" if self.passed else "FAIL"
    def continuous(self, at):
        if self.age_days(at) > self.valid_for_days: return "STALE"
        return "PASS" if self.passed else "FAIL"

TESTS = [
 ControlTest("AC-1", True,  "act chain sampled from gateway logs", now -   3*DAY),
 ControlTest("AC-2", True,  "delegation refusal regression suite", now -   9*DAY),
 ControlTest("SB-1", True,  "egress denial evidence",              now -  45*DAY),
 ControlTest("SB-2", True,  "approval gate screenshot",            now - 210*DAY),
 ControlTest("EV-1", True,  "audit sample of 50 agent actions",    now -   5*DAY),
 ControlTest("DR-1", False, "drift alerting not deployed",         now),
]
REQUIRED = ["AC-1", "AC-2", "SB-1", "SB-2", "EV-1", "DR-1", "EV-2", "ST-1"]

print(f"{'control':9s}{'age (days)':>12}{'point-in-time':>16}{'continuous':>13}")
print("-" * 52)
by_id = {t.cid: t for t in TESTS}
for cid in REQUIRED:
    t = by_id.get(cid)
    if t is None:
        print(f"{cid:9s}{'—':>12}{'(not tested)':>16}{'NO EVIDENCE':>13}")
        continue
    print(f"{cid:9s}{t.age_days(now):>12.0f}{t.point_in_time(now):>16}{t.continuous(now):>13}")

def posture(tests, required, at, mode):
    by_id = {t.cid: t for t in tests}
    passing = 0
    for cid in required:
        t = by_id.get(cid)
        if t is None: continue
        state = t.point_in_time(at) if mode == "point-in-time" else t.continuous(at)
        passing += state == "PASS"
    return passing, round(passing/len(required), 3)

for mode in ("point-in-time", "continuous"):
    n, pct = posture(TESTS, REQUIRED, now, mode)
    print(f"{mode:16s} {n}/{len(REQUIRED)} controls passing = {pct:.0%}")

print("\nThe difference is entirely SB-1 and SB-2, which nobody did anything")
print("wrong to. Time simply passed, and the agent they were tested against")
print("has had two model upgrades since.")

DRIFT_RATE = {          # observed TVD/day for what each control depends on
 "AC-1": 0.0005,        # identity model changes slowly
 "AC-2": 0.0005,
 "SB-1": 0.0020,        # egress needs change with new integrations
 "SB-2": 0.0090,        # tool manifests change weekly
 "EV-1": 0.0010,
 "DR-1": 0.0090,
}
TOLERANCE = 0.25

def window(cid):
    r = DRIFT_RATE.get(cid)
    return int(TOLERANCE / r) if r else 90

print(f"{'control':9s}{'drift/day':>12}{'window (days)':>15}{'current age':>13}{'state':>9}")
print("-" * 60)
for cid in REQUIRED:
    t = by_id.get(cid)
    w = window(cid)
    if t is None:
        print(f"{cid:9s}{'—':>12}{w:>15}{'—':>13}{'NO EVIDENCE':>9}")
        continue
    t.valid_for_days = w
    print(f"{cid:9s}{DRIFT_RATE.get(cid, 0):>12.4f}{w:>15}{t.age_days(now):>13.0f}"
          f"{t.continuous(now):>9}")

n, pct = posture(TESTS, REQUIRED, now, "continuous")
print(f"\nwith drift-derived windows: {n}/{len(REQUIRED)} = {pct:.0%} currently evidenced")
assert pct < 0.6
print("\nSB-2 tests a tool manifest that changes weekly; a 210-day-old screenshot")
print("cannot evidence it. Saying so is the control, not a criticism of anyone.")

## What you just proved

The skill loads and reports its shape, and its failure modes are the governance lesson stated as engineering: the relying party must fail closed on a missing attestation, because reading absence as a pass is exactly the annual-test habit arriving in a new format. Drift against the digest and the manifest is what re-triggers it, not the calendar.

## Your turn

Pick your three most important AI controls and set a freshness window for each from the observed change rate of what it tests. Then recompute your posture. The number will drop, and it will be the first honest one you have had.

---

**Next → [E1.2 · Building the AI and agent inventory](https://spbreed.github.io/cyber-commons/lessons/E1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*